# Test Angular Spectrum Fix v2

Compare:
- v1: Fraunhofer formula + 20 pixels across aperture
- v2: Fraunhofer formula + 200 pixels across aperture

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special import j1

from monte_carlo.angular_spectrum_fixed import AngularSpectrumSimulatorFixed
from monte_carlo.angular_spectrum_fixed2 import AngularSpectrumSimulatorFixed2
from monte_carlo import metrics

sns.set_theme(style="whitegrid", font_scale=1.5)

import os
output_dir = '../data/angular_spectrum_fix_v2_test'
os.makedirs(output_dir, exist_ok=True)

## Setup Parameters

In [ ]:
# Simulation parameters
n_photons = 1000000
wavelength = 0.6328  # microns
focal_length = 10.0  # mm
numerical_aperture = 0.1
n_medium = 1.0
random_seed = 42
fft_size = 2048

airy_radius = 1.22 * wavelength / numerical_aperture

print(f"N photons: {n_photons:,}")
print(f"Wavelength: {wavelength} μm")
print(f"NA: {numerical_aperture}")
print(f"Focal length: {focal_length} mm")
print(f"FFT size: {fft_size}")
print(f"First Airy zero: {airy_radius:.4f} μm")

## Run Both Versions

In [ ]:
print("Running v1 (20 pixels across aperture)...")
sim_v1 = AngularSpectrumSimulatorFixed(
    n_photons=n_photons,
    wavelength=wavelength,
    focal_length=focal_length,
    numerical_aperture=numerical_aperture,
    n_medium=n_medium,
    random_seed=random_seed,
    fft_size=fft_size
)
results_v1 = sim_v1.propagate()
x_v1, y_v1, _ = results_v1['focal_positions']
print(f"  Got {len(x_v1):,} photons")

print("\nRunning v2 (200 pixels across aperture)...")
sim_v2 = AngularSpectrumSimulatorFixed2(
    n_photons=n_photons,
    wavelength=wavelength,
    focal_length=focal_length,
    numerical_aperture=numerical_aperture,
    n_medium=n_medium,
    random_seed=random_seed,
    fft_size=fft_size
)
results_v2 = sim_v2.propagate()
x_v2, y_v2, _ = results_v2['focal_positions']
print(f"  Got {len(x_v2):,} photons")

## Compute Theoretical Airy Pattern

In [ ]:
# Radial bins
plot_range = 2 * airy_radius
r_bins = np.linspace(0, plot_range, 300)
r_centers = (r_bins[:-1] + r_bins[1:]) / 2
bin_areas = np.pi * (r_bins[1:]**2 - r_bins[:-1]**2)

# Theoretical Airy pattern
k_theory = 2 * np.pi * numerical_aperture / wavelength
kr_theory = k_theory * r_centers
airy_theory = np.ones_like(kr_theory)
nonzero = kr_theory != 0
airy_theory[nonzero] = (2 * j1(kr_theory[nonzero]) / kr_theory[nonzero])**2

print(f"Theoretical Airy FWHM: {1.22 * wavelength / numerical_aperture:.4f} μm (approx)")

## Compute Radial Profiles

In [ ]:
# v1
r_v1 = np.sqrt(x_v1**2 + y_v1**2)
hist_v1, _ = np.histogram(r_v1, bins=r_bins)
intensity_v1 = hist_v1 / bin_areas
intensity_v1 = intensity_v1 / np.max(intensity_v1)

# v2
r_v2 = np.sqrt(x_v2**2 + y_v2**2)
hist_v2, _ = np.histogram(r_v2, bins=r_bins)
intensity_v2 = hist_v2 / bin_areas
intensity_v2 = intensity_v2 / np.max(intensity_v2)

# Compute RMSE
rmse_v1 = metrics.rmse(intensity_v1, airy_theory)
rmse_v2 = metrics.rmse(intensity_v2, airy_theory)

print(f"\nRMSE Comparison:")
print(f"  v1 (20 pixels):   {rmse_v1:.6f}")
print(f"  v2 (200 pixels):  {rmse_v2:.6f}")
print(f"  Improvement: {(rmse_v1 - rmse_v2) / rmse_v1 * 100:.1f}%")

## Check K-space Distribution Quality

In [ ]:
# Sample k-vectors from both
kx_v1, ky_v1 = sim_v1.sample_angular_spectrum()
k_rho_v1 = np.sqrt(kx_v1**2 + ky_v1**2)

kx_v2, ky_v2 = sim_v2.sample_angular_spectrum()
k_rho_v2 = np.sqrt(kx_v2**2 + ky_v2**2)

# Theoretical jinc^2
k_rho_theory = np.linspace(0, sim_v1.k_max, 200)
kr = k_rho_theory * sim_v1.aperture_radius
jinc = np.ones_like(kr)
nonzero = kr != 0
jinc[nonzero] = 2 * j1(kr[nonzero]) / kr[nonzero]
intensity_theory = jinc**2

# Histograms
k_bins = np.linspace(0, sim_v1.k_max, 100)
k_centers = (k_bins[:-1] + k_bins[1:]) / 2
k_bin_areas = np.pi * (k_bins[1:]**2 - k_bins[:-1]**2)

hist_k_v1, _ = np.histogram(k_rho_v1, bins=k_bins)
intensity_k_v1 = hist_k_v1 / k_bin_areas
intensity_k_v1 = intensity_k_v1 / np.max(intensity_k_v1)

hist_k_v2, _ = np.histogram(k_rho_v2, bins=k_bins)
intensity_k_v2 = hist_k_v2 / k_bin_areas
intensity_k_v2 = intensity_k_v2 / np.max(intensity_k_v2)

# Compute RMSE in k-space
from scipy.interpolate import interp1d
theory_interp = interp1d(k_rho_theory, intensity_theory, bounds_error=False, fill_value=0)
intensity_theory_at_k = theory_interp(k_centers)
rmse_k_v1 = metrics.rmse(intensity_k_v1, intensity_theory_at_k)
rmse_k_v2 = metrics.rmse(intensity_k_v2, intensity_theory_at_k)

print(f"\nK-space RMSE:")
print(f"  v1: {rmse_k_v1:.6f}")
print(f"  v2: {rmse_k_v2:.6f}")
print(f"  Improvement: {(rmse_k_v1 - rmse_k_v2) / rmse_k_v1 * 100:.1f}%")

## Plot Comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Top left: K-space distribution
ax = axes[0, 0]
ax.plot(k_rho_theory, intensity_theory, 'k-', linewidth=4, alpha=0.5, label='Theory: jinc²', zorder=10)
ax.plot(k_centers, intensity_k_v1, 'r-', linewidth=2, alpha=0.7, label=f'v1 (RMSE={rmse_k_v1:.4f})')
ax.plot(k_centers, intensity_k_v2, 'g-', linewidth=2, alpha=0.7, label=f'v2 (RMSE={rmse_k_v2:.4f})')
ax.set_xlabel('k_ρ (mm^-1)', fontsize=14)
ax.set_ylabel('Normalized Intensity', fontsize=14)
ax.set_title('K-space Distribution', fontsize=16, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

# Top right: Focal plane profiles
ax = axes[0, 1]
ax.plot(r_centers, airy_theory, 'k-', linewidth=4, alpha=0.5, label='Airy (theory)', zorder=10)
ax.plot(r_centers, intensity_v1, 'r-', linewidth=2, alpha=0.7, label=f'v1 (RMSE={rmse_v1:.4f})')
ax.plot(r_centers, intensity_v2, 'g-', linewidth=2, alpha=0.7, label=f'v2 (RMSE={rmse_v2:.4f})')
ax.set_xlabel('Radial Distance (μm)', fontsize=14)
ax.set_ylabel('Normalized Intensity', fontsize=14)
ax.set_title('Focal Plane Distribution', fontsize=16, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_xlim([0, plot_range])

# Bottom left: Zoomed k-space
ax = axes[1, 0]
zoom_k = 0.3 * sim_v1.k_max
zoom_mask_k = k_centers < zoom_k
ax.plot(k_rho_theory[k_rho_theory < zoom_k], intensity_theory[k_rho_theory < zoom_k], 'k-', linewidth=4, alpha=0.5, label='Theory', zorder=10)
ax.plot(k_centers[zoom_mask_k], intensity_k_v1[zoom_mask_k], 'r-', linewidth=2, alpha=0.7, label='v1')
ax.plot(k_centers[zoom_mask_k], intensity_k_v2[zoom_mask_k], 'g-', linewidth=2, alpha=0.7, label='v2')
ax.set_xlabel('k_ρ (mm^-1)', fontsize=14)
ax.set_ylabel('Normalized Intensity', fontsize=14)
ax.set_title('K-space (Zoomed)', fontsize=16, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

# Bottom right: Zoomed focal plane
ax = axes[1, 1]
zoom_r = airy_radius
zoom_mask_r = r_centers < zoom_r
ax.plot(r_centers[zoom_mask_r], airy_theory[zoom_mask_r], 'k-', linewidth=4, alpha=0.5, label='Theory', zorder=10)
ax.plot(r_centers[zoom_mask_r], intensity_v1[zoom_mask_r], 'r-', linewidth=2, alpha=0.7, label='v1')
ax.plot(r_centers[zoom_mask_r], intensity_v2[zoom_mask_r], 'g-', linewidth=2, alpha=0.7, label='v2')
ax.set_xlabel('Radial Distance (μm)', fontsize=14)
ax.set_ylabel('Normalized Intensity', fontsize=14)
ax.set_title('Focal Plane (Zoomed)', fontsize=16, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{output_dir}/comparison_v2.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

In [ ]:
print("="*80)
print("COMPARISON: v1 vs v2 Angular Spectrum Fix")
print("="*80)
print(f"\nv1 (20 pixels across aperture):")
print(f"  K-space RMSE:      {rmse_k_v1:.6f}")
print(f"  Focal plane RMSE:  {rmse_v1:.6f}")
print(f"\nv2 (200 pixels across aperture):")
print(f"  K-space RMSE:      {rmse_k_v2:.6f}")
print(f"  Focal plane RMSE:  {rmse_v2:.6f}")
print(f"\n{'='*80}")
print(f"K-space improvement:     {(rmse_k_v1 - rmse_k_v2) / rmse_k_v1 * 100:.1f}%")
print(f"Focal plane improvement: {(rmse_v1 - rmse_v2) / rmse_v1 * 100:.1f}%")
print(f"{'='*80}")